In [1]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

with tf.device("/GPU:0"):
    a = tf.random.normal((2048, 2048))
    b = tf.random.normal((2048, 2048))
    result = tf.matmul(a, b)

print("Result device:", result.device)

TensorFlow: 2.10.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Result device: /job:localhost/replica:0/task:0/device:GPU:0


In [2]:
import time
import subprocess
import tensorflow as tf
from tensorflow.keras import layers, models

def print_gpu_info():
    print(f"TensorFlow Version: {tf.__version__}")
    gpus = tf.config.list_physical_devices('GPU')
    print(f"Detected GPUs: {gpus}")
    
    # On Colab (Linux), nvidia-smi prints the exact GPU name (e.g., Tesla T4)
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            stderr=subprocess.DEVNULL
        ).decode().strip()
        print(f"Hardware GPU: {out}")
    except Exception:
        print("Hardware GPU: DirectML Device (Local Windows)")

def build_cnn(img_size=128, num_classes=10):
    return models.Sequential([
        layers.Input(shape=(img_size, img_size, 3)),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dense(num_classes)
    ])

def run_gpu_benchmark(batch_size=64, img_size=128, steps=50, warmup=10):
    gpus = tf.config.list_physical_devices('GPU')
    if not gpus:
        print("ERROR: No GPU detected! Switch runtime to T4 GPU in Colab or activate your DirectML venv.")
        return

    with tf.device('/GPU:0'):
        model = build_cnn(img_size=img_size)
        optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

        x = tf.random.normal((batch_size, img_size, img_size, 3))
        y = tf.random.uniform((batch_size,), minval=0, maxval=10, dtype=tf.int32)

        @tf.function
        def train_step(images, labels):
            with tf.GradientTape() as tape:
                logits = model(images, training=True)
                loss = loss_fn(labels, logits)
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            return loss

        # 1. Warmup (compiles CUDA/cuDNN kernels on T4 or DirectX 12 shaders on RX 580)
        print(f"Warming up ({warmup} steps)...")
        for _ in range(warmup):
            loss = train_step(x, y)
        _ = loss.numpy()  # Force host-device sync

        # 2. Timed Benchmark
        print(f"Benchmarking ({steps} steps, Batch Size={batch_size}, Image={img_size}x{img_size})...")
        start_time = time.perf_counter()
        for _ in range(steps):
            loss = train_step(x, y)
        _ = loss.numpy()  # Force GPU to complete all queued work before stopping clock
        elapsed = time.perf_counter() - start_time

    total_images = batch_size * steps
    throughput = total_images / elapsed
    ms_per_step = (elapsed / steps) * 1000

    print("\n--- RESULTS ---")
    print(f"Total Time : {elapsed:.2f} seconds")
    print(f"Step Time  : {ms_per_step:.1f} ms/step")
    print(f"Throughput : {throughput:.1f} images/sec")

if __name__ == "__main__":
    print_gpu_info()
    print("-" * 40)
    run_gpu_benchmark(batch_size=64, img_size=128, steps=50, warmup=10)

TensorFlow Version: 2.10.0
Detected GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Hardware GPU: DirectML Device (Local Windows)
----------------------------------------
Warming up (10 steps)...
Benchmarking (50 steps, Batch Size=64, Image=128x128)...

--- RESULTS ---
Total Time : 27.10 seconds
Step Time  : 541.9 ms/step
Throughput : 118.1 images/sec
